# 200-Level Data Compression

Explore two lossless compression ideas from this repository: the Burrows-Wheeler transform (BWT) and Huffman coding. Lossless means decoding recreates the exact original text.

The repository implementations are `BurrowsWheelerTransform` and `HuffmanCompressor` in `Algorithms/200-level/DataCompression/`.

## Before you start

Compression removes redundancy, not meaning. For example, `AAAAAB` contains repeated information and is easier to describe compactly than `QXJMRB`.

- **BWT** rearranges text so repeated characters tend to become neighbors. It needs an index to undo the rearrangement.
- **Huffman coding** gives common characters shorter binary codes and rare characters longer codes.
- Neither technique encrypts data: their purpose is smaller representations, not secrecy.

## 1. Burrows-Wheeler transform

For a string of length `n`, create all `n` rotations, sort them, and take the last character from each sorted rotation. Store where the original string appears in the sorted list.

**Time complexity:** O(n^2 log n) in this straightforward educational version, because it builds and sorts full strings.

**Space complexity:** O(n^2) for the rotation table.

In [ ]:
def bwt_encode(text):
    """Return the BWT last column and the original row index."""
    if not text:
        return "", 0

    rotations = [text[index:] + text[:index] for index in range(len(text))]
    rotations.sort()
    encoded = "".join(rotation[-1] for rotation in rotations)
    return encoded, rotations.index(text)


def bwt_decode(encoded, original_index):
    """Rebuild the sorted rotation table and return its original row."""
    if not encoded:
        return ""

    rotations = [""] * len(encoded)
    for _ in encoded:
        rotations = sorted(encoded[index] + rotations[index] for index in range(len(encoded)))

    return rotations[original_index]


sample = "BANANA"
encoded, row = bwt_encode(sample)
print(f"Original: {sample}")
print(f"BWT:      {encoded}, original row: {row}")
print(f"Decoded:  {bwt_decode(encoded, row)}")
assert bwt_decode(encoded, row) == sample

**Observe:** The BWT output is not necessarily shorter than the input. Its value is that it often groups similar symbols together, which makes a following compression step more effective.

## 2. Huffman coding

Huffman coding repeatedly joins the two least-frequent nodes into a binary tree. Going left adds `0`; going right adds `1`. The path to a character becomes that character's code.

A prefix-free code has no code that is the beginning of another code. That property lets a decoder know exactly where each character ends.

**Time complexity:** O(n log k) with a priority queue, where `n` is the text length and `k` is the number of distinct symbols.

**Space complexity:** O(n + k).

In [ ]:
from collections import Counter
from heapq import heappop, heappush
from itertools import count


def huffman_codes(text):
    """Build a character-to-bitstring table for non-empty text."""
    if not text:
        return {}

    sequence = count()
    heap = []
    for character, frequency in Counter(text).items():
        heappush(heap, (frequency, next(sequence), character))

    if len(heap) == 1:
        return {heap[0][2]: "0"}

    while len(heap) > 1:
        left_frequency, _, left = heappop(heap)
        right_frequency, _, right = heappop(heap)
        heappush(heap, (left_frequency + right_frequency, next(sequence), (left, right)))

    _, _, root = heap[0]
    codes = {}

    def visit(node, prefix):
        if isinstance(node, str):
            codes[node] = prefix
            return

        left, right = node
        visit(left, prefix + "0")
        visit(right, prefix + "1")

    visit(root, "")
    return codes


def huffman_encode(text, codes):
    return "".join(codes[character] for character in text)


def huffman_decode(bits, codes):
    reverse_codes = {code: character for character, code in codes.items()}
    decoded = []
    current = ""
    for bit in bits:
        current += bit
        if current in reverse_codes:
            decoded.append(reverse_codes[current])
            current = ""

    if current:
        raise ValueError("The bit string ends in an incomplete Huffman code.")
    return "".join(decoded)

In [ ]:
message = "beep boop beer!"
codes = huffman_codes(message)
compressed_bits = huffman_encode(message, codes)
decoded_message = huffman_decode(compressed_bits, codes)

print("Codes:", dict(sorted(codes.items())))
print("Original bits (8-bit characters):", len(message) * 8)
print("Huffman data bits:", len(compressed_bits))
print("Round trip works:", decoded_message == message)

assert decoded_message == message
assert huffman_decode("000", huffman_codes("AAAA")) == "AAA"

## Deeper Practice (Same-Level)

- **BWT round-trip** - _same-level stretch_: Test `bwt_encode` and `bwt_decode` with an empty string, one character, and repeated characters.
- **Frequency reasoning** - _same-level stretch_: Compare Huffman codes for a message with one very common character and several rare ones. Which codes become shortest?
- **Prefix check** - _same-level advanced_: Write a function that verifies no code in a Huffman table is a prefix of another.
- **Compression accounting** - _same-level advanced_: Include the size of the code table in your comparison. When is Huffman coding not worthwhile for a short message?